# 03 — Metamodel Joint Inference

Build the joint metamodel coupling all 4 surrogates and run Bayesian inference.

**Prerequisites**: Trained surrogates from notebook 02.

## What `meta sample` actually does — read this before the numbers

It is easy to read "metamodel" and "sample" and assume MCMC conditioning on data.
That is **not** what this command does today, and the difference changes how you
should read every number below.

From the framework's own docstring (`meta/sampling.py`):

> *Generate samples from a metamodel IR using prior draws and coupling transforms
> … coupling constraints are applied as a post-draw transform approximation.*

Concretely, per draw it:

1. draws every variable independently from its prior, then
2. overwrites each **coupled target** with `transform(source)` (+ noise, for a
   soft link).

Two consequences worth stating plainly:

- **The four surrogate likelihoods are not evaluated during sampling.** The
  sampler is called with an empty surrogate map, so those factors contribute
  nothing. The surrogates are still built into the IR and still validated — they
  are simply not yet what drives the draws.
- **There is no conditioning, so "posterior" is a misnomer.** Nothing is being
  updated by data. What you get is *forward uncertainty propagation* through the
  coupled chain — which is a real and useful thing, and is what this notebook
  demonstrates.

This is worth knowing rather than glossing: a run that reported a "posterior"
identical to the prior would look like a modelling failure, when in fact it is
the documented behaviour of a baseline sampler.

## Why the couplings had to be fixed first

Until now this spec carried three couplings that looked meaningful and were not:

```json
{"kind": "gaussian_link", "source": "contact_fraction",
 "target": "contact_fraction", "transform": {"kind": "identity"}}
```

Source and target are the **same variable**, so the compiler's residual is
`target - target == 0` for every draw. Three provable no-ops. On top of that, five
variables had no prior at all and silently defaulted to `N(0, 1)` — which for
`depletion_width_nm` meant the sampler was drawing *negative nanometres*. Between
them, that is why every variable came back at mean ~0, sd ~1.

The spec now carries one coupling, and it is the mechanism the paper is about
rather than a curve fit:

$$\text{cd45\_boundary} = \frac{\text{cd45\_bulk}}{1 - \text{contact\_fraction}}$$

Excluding CD45 from the tight contact concentrates it in the remaining area — the
kinetic-segregation step itself. It is declared as a `deterministic` affine link
(α = 588.81, β = 277.30), the linearisation of that expression over the
contact-fraction range the sweeps actually produced, `[0.087, 0.442]`. Deterministic
matters: the docstring notes the post-draw approximation is *exact* for
deterministic transforms and biased for soft links.

The other variables' priors are now the mean and spread **observed in notebook 02's
sweeps**, so a draw is a plausible state of this system rather than a standard normal.

## One more limitation, visible in the numbers below

`_prior_params` accepts only `kind: "normal"`, and draws come from `rng.normal`.
Gaussian priors are therefore the only ones available — which is fine for
`depletion_width_nm` (a positive length, comfortably far from zero) and wrong in
principle for a **bounded** quantity.

`ptcr_fraction` is the case to watch. The sweeps put it at mean 0.86 with sd 0.34,
because the model saturates near 1 for much of the design. No Gaussian can express
"mostly near 1, never above 1", so a fraction of the draws land above 1.0. The
self-check reports that fraction rather than hiding it, and asserts only what a
Gaussian *can* honestly deliver — a median inside the physical range.

The fix is not to shrink the prior until the leak disappears; that would understate
a spread the data really shows. It is a bounded prior (Beta, or a truncated normal),
which this sampler does not yet support. Worth knowing before you read a fraction
near its ceiling as if it were calibrated.


> **This notebook needs the `bayesian-metamodeling` framework.**
> It drives the `bayesmm` CLI, unlike the kinetic-segregation series under
> `notebooks/models/kinetic_segregation/`, which needs only numpy and the compiled
> model. If `bayesmm` is missing, install the framework
> (`pip install -e .` from the parent repo, or `pip install bayesian-metamodeling`)
> and restart the kernel.
>
> New here? Start at [`Tutorial_0_Start_Here.ipynb`](Tutorial_0_Start_Here.ipynb).

## Learning aims
- **Primary**: build the metamodel IR from coupled surrogates and sample the joint
  posterior.
- **Secondary scientific**: explain what a coupling asserts, and what "joint" buys you
  over four separate posteriors.

## What coupling means

Each partial model has its own posterior. Independently, their joint distribution is
just a product — nothing relates them. A **coupling** states that two variables in
different models are the same physical quantity, or are related by a known transform.

`specs/metamodel.tcr_signaling.json` couples `depletion_width_nm` with a
`gaussian_link` of sigma 10 nm: *these should agree, to within 10 nm*. That constraint
propagates — data that sharpens one model's posterior now sharpens its neighbours too.

**Requires `pymc`.** Sampling is the slow step; the draw counts here are deliberately
modest for interactive use, and production values are noted inline.

In [ ]:
import json
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find this repo, instead of assuming a fixed depth.

    The previous version walked up a fixed number of levels from the working
    directory, which silently assumed a submodule checkout and broke in a
    standalone clone or from any other directory. Searching for a landmark is
    robust to both.
    """
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "models" / "kinetic_segregation" / "CMakeLists.txt").is_file():
            return cand
    raise RuntimeError(f"could not locate the tcr_signaling repo above {here}")


ROOT = find_repo_root()
SPECS = ROOT / "specs"
# The metamodel spec every cell below drives. Its assignment was missing, so
# the first `meta build` cell raised NameError — unnoticed because no CI job
# executed these notebooks until `Submodule notebooks CI` was added.
META_SPEC = SPECS / "metamodel.tcr_signaling.json"
print(f"repo root: {ROOT}")

# These notebooks drive the bayesian-metamodeling CLI. Report clearly if absent.
HAVE_BAYESMM = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "--version"], capture_output=True, text=True
).returncode == 0
print("bayesmm:", "available" if HAVE_BAYESMM else "NOT INSTALLED -- see the banner above")


## Build metamodel IR

In [ ]:
r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "meta", "build", str(META_SPEC)],
                   cwd=str(ROOT), capture_output=True, text=True)
print("Return code:", r.returncode)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)

## Sample from the joint posterior

In [ ]:
r = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "meta", "sample", str(META_SPEC), "--draws", "2000", "--tune", "1000"],
    cwd=str(ROOT), capture_output=True, text=True
)
print("Return code:", r.returncode)
print(r.stdout[:500])
if r.returncode != 0:
    print("STDERR:", r.stderr[:500])

## Inspect posterior samples

In [ ]:
r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "meta", "list"], cwd=str(ROOT),
                   capture_output=True, text=True)
print(r.stdout)

## Final check

In [ ]:
# Self-check: the metamodel propagated uncertainty through a real coupling, and the
# variables are physically plausible.
#
# The previous assertion was `assert ROOT.is_dir()`. It passed while every variable
# was N(0,1) — including a depletion width drawing negative nanometres.
import json as _json
import numpy as _np

_irs = sorted((ROOT / "tmp/metamodel_ir").glob("*/ir.json"), key=lambda q: q.stat().st_mtime)
assert _irs, "no metamodel IR was built — Step 1's `meta build` did not run"
_ir = _json.loads(_irs[-1].read_text())
_kinds = {}
for _f in _ir["factors"]:
    _kinds[_f["kind"]] = _kinds.get(_f["kind"], 0) + 1
print(f"  IR: {len(_ir['variables'])} variables, {len(_ir['factors'])} factors {_kinds}")
assert _kinds.get("surrogate_likelihood") == 4, "expected one surrogate likelihood per partial model"
assert _kinds.get("coupling", 0) >= 1, "no coupling: four independent models, not a metamodel"

# No coupling may be a self-link. That is not a style point — the compiler computes
# `residual = target - transform(source)`, so source == target under identity is
# identically zero and the factor does nothing at all.
for _f in _ir["factors"]:
    if _f["kind"] != "coupling":
        continue
    assert _f["source"] != _f["target"], (
        f"coupling {_f['source']} -> {_f['target']} is a self-link and contributes nothing"
    )

_s = sorted((ROOT / "tmp/metamodel_samples").glob("*/samples_dataset.json"),
            key=lambda q: q.stat().st_mtime)
assert _s, "no samples — Step 2's `meta sample` did not run"
_v = _json.loads(_s[-1].read_text())["variables"]

# Physical plausibility. Each of these failed under the old N(0,1) default.
# Strict for quantities a Gaussian prior can represent honestly; reported-only for
# bounded ones, where the sampler's normal-only priors must leak (see above).
_bounds = {
    "contact_fraction":      (0.0, 1.0, True),      # fraction, but far from its ceiling
    "depletion_width_nm":    (0.0, 5000.0, True),   # nanometres, positive
    "cd45_boundary_density": (0.0, 5000.0, True),   # molecules/um^2, positive
    "ptcr_fraction":         (0.0, 1.0, False),     # saturates near 1: a Gaussian must spill
}
for _name, (_lo, _hi, _strict) in _bounds.items():
    _d = _np.asarray(_v[_name], dtype=float).ravel()
    assert _d.size > 100, f"{_name}: only {_d.size} draws"
    assert _np.all(_np.isfinite(_d)), f"{_name}: non-finite draws"
    _frac = float(_np.mean((_d >= _lo) & (_d <= _hi)))
    if _strict:
        assert _frac > 0.95, (
            f"{_name}: only {_frac:.0%} of draws lie in the physical range [{_lo}, {_hi}] "
            f"(mean={_d.mean():.4g}, sd={_d.std():.4g}) — is its prior missing? An absent "
            "prior defaults to N(0,1), which is how this spec once drew negative nanometres."
        )
    else:
        # Bounded and near its ceiling: assert only what a normal prior can deliver.
        _med = float(_np.median(_d))
        assert _lo <= _med <= _hi, f"{_name}: median {_med:.4g} outside [{_lo}, {_hi}]"
    _note = "" if _strict else "  <- normal prior on a bounded quantity; see note above"
    print(f"  {_name:22} mean={_d.mean():9.4g} sd={_d.std():8.4g}  {_frac:.0%} in range{_note}")

# The propagation itself: a deterministic affine link must reproduce exactly.
_cf = _np.asarray(_v["contact_fraction"], dtype=float).ravel()
_cd = _np.asarray(_v["cd45_boundary_density"], dtype=float).ravel()
_r = float(_np.corrcoef(_cf, _cd)[0, 1])
assert _r > 0.999, (
    f"corr(contact_fraction, cd45_boundary_density) = {_r:.4f}; a deterministic link "
    "should reproduce its source exactly. The coupling is not propagating."
)
print(f"\n  corr(contact_fraction -> cd45_boundary_density) = {_r:+.6f}  (deterministic link)")
print(f"\n[NB03 self-check OK] 4 surrogates in the IR, 1 real coupling, propagation verified")